<a href="https://colab.research.google.com/github/kurubaLab/agentic-ai/blob/main/Hello_Agent_CSV_FAQ_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gdown
!pip install langchain_openai
!pip install langchain_experimental


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [1]:
import gdown
import os

# ======================================
## Step 1: Download all of the CSV files
# ======================================

files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

print(f"--- Downloading {len(files_to_download)} CSV Files from Google Drive ---")
for file_name, file_url in files_to_download.items() :
  if not os.path.exists(file_name) :
    gdown.download(file_url, file_name, quiet=True, fuzzy=True)
    print(f"  -- Downloaded {file_name}")
  else:
    print(f"  -- Download Skipped. File {file_name} already exists")

--- Downloading 4 CSV Files from Google Drive ---
  -- Download Skipped. File saas_docs.csv already exists
  -- Download Skipped. File credit_card_terms.csv already exists
  -- Download Skipped. File hospital_policy.csv already exists
  -- Download Skipped. File ecommerce_faqs.csv already exists


In [2]:
import pandas as pd

# ===============================================
## Step 2: Load the CSV file content using Pandas
# ===============================================
dataframes = []
loaded_filenames = []

try:
  for file_name in files_to_download.keys() :
    df = pd.read_csv(file_name)
    dataframes.append(df)
    loaded_filenames.append(file_name)
    print(f"  -- Loaded {len(df)} rows from {file_name}")

except Exception as e:
  print(f"Error: {e}")

  -- Loaded 15 rows from saas_docs.csv
  -- Loaded 15 rows from credit_card_terms.csv
  -- Loaded 15 rows from hospital_policy.csv
  -- Loaded 15 rows from ecommerce_faqs.csv


In [9]:
print(dataframes)

[   Doc ID         Feature  Plan Level  \
0   S-501  API Rate Limit        Free   
1   S-502  API Rate Limit  Enterprise   
2   S-503      User Roles         All   
3   S-504     Data Export         Pro   
4   S-505    Integrations         Pro   
5   S-506             SSO  Enterprise   
6   S-507    File Storage        Free   
7   S-508    File Storage  Enterprise   
8   S-509      Audit Logs  Enterprise   
9   S-510   Custom Fields         Pro   
10  S-511        Webhooks         Pro   
11  S-512         Support        Free   
12  S-513         Support  Enterprise   
13  S-514      Encryption         All   
14  S-515      Mobile App         All   

                                          Description Technical Limit  \
0   Users on the Free tier are limited to 1,000 AP...    1000 req/day   
1   Enterprise workspaces have a dedicated pool of...      1M req/day   
2   Admins have full access to billing and workspa...             NaN   
3   Data can be exported in CSV, JSON, or XML form

In [6]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import sys


# ================================
## Step 3: Initialize the AI Agent
# ================================

# Step 3a - Obtain the OpenAI API key
my_api_key = userdata.get('OPENAI_API_KEY')
print(f'Obtained OpenAI API Key - {my_api_key[0:10]}')

#Step 3b - Initialize the rules
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

try:
  #Step 3c - Initialize the LLM
  llm = ChatOpenAI(
      openai_api_key=my_api_key,
      temperature=0,
      model_name='gpt-4o-mini'
  )

  #Step 3d - Create Pandas Agent
  agent = create_pandas_dataframe_agent(
      llm,
      dataframes,
      verbose=True,
      agent_type="openai-functions",
      system_message=system_prompt,
      allow_dangerous_code=True
  )
  print("\nAI Agent is ready! You can ask questions across ALL files.")
  print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()

Obtained OpenAI API Key - sk-proj-rN

AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'system_message': '\nYou are a smart data assistant capable of reading multiple CSV files.\n- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.\n- When asked a question, determine which DataFrame is most relevant.\n- Do NOT answer from general knowledge.\n- Answer in plain English.\n'} which are no longer supported.
  warnings.warn(


In [10]:
#====================
## Create Chat Loop
#====================
while True:
  try:
    user_input = input("Enter your question (or 'exit' to quit): ")
    if user_input.lower() == 'exit':
      print("Exiting the program.")
      break
    else:
      query_to_agent = f"Question: {user_input}"
      response = agent.run(query_to_agent)
      print(f"Response: {response}")
  except Exception as e:
    print(f"Error: {e}")
    sys.exit()

Enter your question (or 'exit' to quit): Tell me about SoundBlast 300


> Entering new AgentExecutor chain...
It seems that "SoundBlast 300" is not mentioned in any of the provided dataframes (df1, df2, df3, df4). If you have specific information or context about "SoundBlast 300," please provide it, and I can help you with that! Alternatively, if you meant to refer to a different product or topic, please clarify.

> Finished chain.
Response: It seems that "SoundBlast 300" is not mentioned in any of the provided dataframes (df1, df2, df3, df4). If you have specific information or context about "SoundBlast 300," please provide it, and I can help you with that! Alternatively, if you meant to refer to a different product or topic, please clarify.
Enter your question (or 'exit' to quit): What about Brewmaster?


> Entering new AgentExecutor chain...
It seems like you are asking about "Brewmaster," but the context is unclear. Could you please provide more details or specify what information 